In [1]:
import numpy as np
import statistics
import random
import gc
import os
import shutil
import cv2
import matplotlib.pyplot as plt
import imutils
from PIL import Image
from scipy import ndimage
# import h5py
import pandas as pd

import json
import copy
import datetime
import sys
from scipy.fftpack import fft2, ifft2, fftshift, ifftshift
import matplotlib.pyplot as plt
%matplotlib inline
import tifffile
from skimage import filters, morphology, segmentation
import imagecodecs
print(imagecodecs.__version__)

2024.9.22


In [2]:
wsl_pyf = r"C:\Users\u9635\pyfolder\EM_analysis/"#"/home/kinoshita/"

In [3]:
import json
import sys
from copy import deepcopy

def filter_small_polygons(input_path: str):
    with open(input_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    # shapesが存在しない場合の安全策
    shapes = data.get("shapes", [])
    filtered = []
    # notp=[]
    for shp in shapes:
        # shape_typeがpolygonのものを対象に、points数が1または2を抽出
        if shp.get("shape_type") == "polygon":
            pts = shp.get("points", [])
            if isinstance(pts, list) and len(pts) in (1, 2):
                filtered.append(shp)

    # 出力JSONは元構造を保ちつつ、shapesだけ差し替え
    # out = deepcopy(data)
    # out["shapes"] = filtered
    return filtered

    # with open(output_path, "w", encoding="utf-8") as f:
    #     json.dump(out, f, ensure_ascii=False, indent=2)

# if __name__ == "__main__":
    # 使い方:
    # python filter_small_polygons.py input.json output.json
    # if len(sys.argv) < 3:
    #     print("Usage: python filter_small_polygons.py <input.json> <output.json>")
    #     sys.exit(1)

    # filter_small_polygons(sys.argv[1], sys.argv[2])


from copy import deepcopy

def remove_small_polygons(input_path: str, output_path: str, min_vertices: int = 3):
    """
    LabelmeのCOCO形式JSONから、頂点数がmin_vertices未満（デフォルト3未満=1点/2点）の
    Polygonを除外し、その他はそのまま書き出します。
    """
    with open(input_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    shapes = data.get("shapes", [])
    filtered_shapes = []
    for shp in shapes:
        pts = shp.get("points", [])
        # 頂点数が3点以上のものだけ残す（1点/2点は除外）
        if isinstance(pts, list) and len(pts) >= min_vertices:
            filtered_shapes.append(shp)

    out = deepcopy(data)
    out["shapes"] = filtered_shapes

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(out, f, ensure_ascii=False, indent=2)



In [5]:
filter_small_polygons(wsl_pyf+"\blind_kinoshita\2\11.json")

[{'label': 'vesicle',
  'points': [[1115.8469945355191, 843.1693989071038]],
  'group_id': None,
  'description': '',
  'shape_type': 'polygon',
  'flags': {},
  'mask': None}]

In [ ]:


remove_small_polygons(wsl_pyf+"\blind_kinoshita\2\11.json", wsl_pyf+"\blind_kinoshita\2\11.json")

In [3]:
unit = "vec"


# #vec
ori_fol = "/train_label_vec2/"
aug_fol =  "/train_label_vecall/"

os.makedirs(wsl_pyf +aug_fol, exist_ok = True)
for n, fn in enumerate(os.listdir(wsl_pyf + ori_fol+"/")):
    # if n > 0:
    #     continue
    print(fn)
    with open(wsl_pyf+ori_fol+fn, 'r') as f:
        data = json.load(f)

    
    for shape in data.get('shapes', []):
        if shape.get('label') == 'vesicle2':
            shape['label'] = 'vesicle'

    # JSON をインデント付きで書き出し
    with open(wsl_pyf+aug_fol+ fn , 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)



Keita #1003.json
Keita #1004.json
Keita #1005.json
Keita #1006.json
Keita #1007.json
Keita #1008.json
Keita #1009.json
Keita #1010.json
Keita #2003.json
Keita #2004.json
Keita #2005.json
Keita #2006.json
Keita #2007.json
Keita #2008.json
Keita #2009.json
Keita #2010.json
Keita #3003.json
Keita #3004.json
Keita #3005.json
Keita #3006.json
Keita #3007.json
Keita #3008.json
Keita #3009.json
Keita #3010.json
Keita_#12003.json
Keita_#12004.json
Keita_#12005.json
Keita_#12006.json
Keita_#12007.json
Keita_#12008.json
Keita_#12009.json
Keita_#12010.json
Keita_#4003.json
Keita_#4004.json
Keita_#4005.json
Keita_#4006.json
Keita_#4007.json
Keita_#4008.json
Keita_#4009.json
Keita_#4010.json
Keita_#5003.json
Keita_#5004.json
Keita_#5005.json
Keita_#5006.json
Keita_#5007.json
Keita_#5008.json
Keita_#5009.json
Keita_#5010.json
Keita_#6003.json
Keita_#6004.json
Keita_#6005.json
Keita_#6006.json
Keita_#6007.json
Keita_#6008.json
Keita_#6009.json
Keita_#6010.json
Keita_#9003.json
Keita_#9004.json
Keita_

In [4]:
unit = "vec"


# #vec
ori_fol = "/train_label_vec2/"
aug_fol =  "/train_label_vec1/"

os.makedirs(wsl_pyf +aug_fol, exist_ok = True)
for n, fn in enumerate(os.listdir(wsl_pyf + ori_fol+"/")):
    # if n > 0:
    #     continue
    print(fn)
    with open(wsl_pyf+ori_fol+fn, 'r') as f:
        data = json.load(f)

    
    # vesicle2 を除いた shapes のリストを作成
    shapes = data.get('shapes', [])
    filtered = [shape for shape in shapes if shape.get('label') != 'vesicle2']
    data['shapes'] = filtered

    # JSON をインデント付きで書き出し
    with open(wsl_pyf+aug_fol+ fn , 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)



Keita #1003.json
Keita #1004.json
Keita #1005.json
Keita #1006.json
Keita #1007.json
Keita #1008.json
Keita #1009.json
Keita #1010.json
Keita #2003.json
Keita #2004.json
Keita #2005.json
Keita #2006.json
Keita #2007.json
Keita #2008.json
Keita #2009.json
Keita #2010.json
Keita #3003.json
Keita #3004.json
Keita #3005.json
Keita #3006.json
Keita #3007.json
Keita #3008.json
Keita #3009.json
Keita #3010.json
Keita_#12003.json
Keita_#12004.json
Keita_#12005.json
Keita_#12006.json
Keita_#12007.json
Keita_#12008.json
Keita_#12009.json
Keita_#12010.json
Keita_#4003.json
Keita_#4004.json
Keita_#4005.json
Keita_#4006.json
Keita_#4007.json
Keita_#4008.json
Keita_#4009.json
Keita_#4010.json
Keita_#5003.json
Keita_#5004.json
Keita_#5005.json
Keita_#5006.json
Keita_#5007.json
Keita_#5008.json
Keita_#5009.json
Keita_#5010.json
Keita_#6003.json
Keita_#6004.json
Keita_#6005.json
Keita_#6006.json
Keita_#6007.json
Keita_#6008.json
Keita_#6009.json
Keita_#6010.json
Keita_#9003.json
Keita_#9004.json
Keita_

In [8]:
img_fol = r"C:\Users\u9635\pyfolder\EM_analysis\train_vec_by_keita/"
new_fol = r"C:\Users\u9635\pyfolder\EM_analysis\train_label_vec/"
os.makedirs(new_fol, exist_ok=True)
prefix = "..\\train\\"

for n, f in enumerate(os.listdir(img_fol)):
    # if n>0:
    #     continue
    with open(img_fol + f, 'r', encoding='utf-8') as file:
        data = json.load(file)
        name = data["imagePath"]
        new_name = prefix + name
        # print(new_name)

        data["imagePath"] = new_name
        print(data["imagePath"])

        with open(new_fol + f, 'w', encoding='utf-8') as file2:
            json.dump(data, file2)
    

..\train\Keita #2003.tif
..\train\Keita #2004.tif
..\train\Keita #2005.tif
..\train\Keita #2006.tif
..\train\Keita #2007.tif
..\train\Keita #2008.tif
..\train\Keita #2009.tif
..\train\Keita #2010.tif
..\train\Keita #3001.tif
..\train\Keita #3002.tif
..\train\Keita #3003.tif
..\train\Keita #3004.tif
..\train\Keita #3005.tif
..\train\Keita #3006.tif
..\train\Keita #3007.tif
..\train\Keita #3008.tif
..\train\Keita #3009.tif
..\train\Keita #3010.tif


In [4]:
def mask_to_polygons(mask: np.ndarray, 
                     approx_eps: float = 1.0
                    ) -> list[list[tuple[int,int]]]:
    """
    2D の bool/0–1 マスクから輪郭を検出し、
    各輪郭を [(x0,y0), (x1,y1), ...] のリストとして返す。

    approx_eps: 輪郭近似の許容誤差 (ピクセル)。0 にするとそのまま生の輪郭点を返す。
    """
    # uint8 化 + 255
    mask_u8 = (mask.astype(np.uint8)) * 255

    # 外周輪郭のみを取得、点群は簡易化 (CHAIN_APPROX_SIMPLE)
    contours, _ = cv2.findContours(
        mask_u8,
        mode = cv2.RETR_EXTERNAL,
        method = cv2.CHAIN_APPROX_SIMPLE
    )

    polys: list[list[tuple[int,int]]] = []
    for cnt in contours:
        # 輪郭点を近似する (省略可)
        if approx_eps > 0:
            cnt = cv2.approxPolyDP(cnt, epsilon=approx_eps, closed=True)
        # reshape & (x,y) タプル化
        coords = cnt.reshape(-1, 2)
        poly = [(int(x), int(y)) for x, y in coords]
        # (面積が小さすぎる輪郭は破棄する場合などフィルタ可)
        if len(poly) >= 3:
            polys.append(poly)

    return polys

In [3]:
#summation of presynapse and vesicle json file
unit1 ="vecall"
unit_syn = "syn"
pre_train = wsl_pyf+"/"+ "train_label_{}".format(unit1)
pre_test  = wsl_pyf+"/"+ "test_label_{}".format(unit1)
pre_train_syn = wsl_pyf+"/"+ "train_label_{}".format(unit_syn)
pre_test_syn  = wsl_pyf+"/"+ "test_label_{}".format(unit_syn)

sum_train_dir = wsl_pyf+"/"+ "train_label_all/"
sum_test_dir = wsl_pyf+"/"+ "test_label_all/"
os.makedirs(sum_train_dir, exist_ok=True)
os.makedirs(sum_test_dir, exist_ok=True)

for i, fname in enumerate(os.listdir(pre_train_syn)):
    # if i>0:
    #     continue
    with open(pre_train_syn + "/" + fname, 'rb') as f:
        data_syn = json.load(f)

    with open(pre_train + "/" + fname, 'rb') as f:
        data_vec = json.load(f)

    data_syn["shapes"].extend(data_vec["shapes"])

    with open(sum_train_dir + "/" + fname, 'w') as f:
        json.dump(data_syn, f)


for i, fname in enumerate(os.listdir(pre_test_syn)):
    # if i>0:
    #     continue
    with open(pre_test_syn + "/" + fname, 'rb') as f:
        data_syn = json.load(f)

    with open(pre_test + "/" + fname, 'rb') as f:
        data_vec = json.load(f)

    data_syn["shapes"].extend(data_vec["shapes"])

    with open(sum_test_dir + "/" + fname, 'w') as f:
        json.dump(data_syn, f)

In [4]:
#split to presynapse and vesicle json file
unit1 ="vecall2"
unit_syn ="syn2"

sum_train_dir = wsl_pyf+"/"+ "labeled_all0/"
sum_test_dir = wsl_pyf+"/"+ "test_label_all0/"

pre_train = wsl_pyf+"/"+ "train_label_{}".format(unit1)
pre_test  = wsl_pyf+"/"+ "test_label_{}".format(unit1)
pre_train_syn = wsl_pyf+"/"+ "train_label_{}".format(unit_syn)
pre_test_syn  = wsl_pyf+"/"+ "test_label_{}".format(unit_syn)
os.makedirs(pre_train, exist_ok=True)
os.makedirs(pre_test, exist_ok=True)
os.makedirs(pre_train_syn, exist_ok=True)
os.makedirs(pre_test_syn, exist_ok=True)


# os.makedirs(sum_train_dir, exist_ok=True)
# os.makedirs(sum_test_dir, exist_ok=True)

for i, fname in enumerate(os.listdir(sum_train_dir)):
    # if i>0:
    #     continue
    print(fname)
    with open(sum_train_dir + "/" + fname, 'rb') as f:
        data = json.load(f)

    base = deepcopy(data)
    base.pop("shapes", None)

    shapes = data.get("shapes", [])
    # filtering
    syn_az_labels = {"synapse", "az"}
    vec_labels = {"vesicle"}

    syn_az_shapes = [s for s in shapes if s.get("label") in syn_az_labels]
    vec_shapes = [s for s in shapes if s.get("label") in vec_labels]

    # 出力データを構築 ("shapes" 以外は全て同じ)
    syn_az_data = deepcopy(base)
    syn_az_data["shapes"] = syn_az_shapes

    vec_data = deepcopy(base)
    vec_data["shapes"] = vec_shapes

    with open(pre_train + "/" + fname, 'w') as f:
        json.dump(vec_data, f)
    with open(pre_train_syn + "/" + fname, 'w') as f:
        json.dump(syn_az_data, f)


# for i, fname in enumerate(os.listdir(sum_test_dir)):
#     # if i>0:
#     #     continue
#     print(fname)
#     with open(sum_test_dir + "/" + fname, 'rb') as f:
#         data = json.load(f)

#     base = deepcopy(data)
#     base.pop("shapes", None)

#     shapes = data.get("shapes", [])
#     # filtering
#     syn_az_labels = {"synapse", "az"}
#     vec_labels = {"vesicle"}

#     syn_az_shapes = [s for s in shapes if s.get("label") in syn_az_labels]
#     vec_shapes = [s for s in shapes if s.get("label") in vec_labels]

#     # 出力データを構築 ("shapes" 以外は全て同じ)
#     syn_az_data = deepcopy(base)
#     syn_az_data["shapes"] = syn_az_shapes

#     vec_data = deepcopy(base)
#     vec_data["shapes"] = vec_shapes

#     with open(pre_test + "/" + fname, 'w') as f:
#         json.dump(vec_data, f)
#     with open(pre_test_syn + "/" + fname, 'w') as f:
#         json.dump(syn_az_data, f)

Keita #1001.json
Keita #1002.json
Keita #1003.json
Keita #1004.json
Keita #1005.json
Keita #1006.json
Keita #1007.json
Keita #1008.json
Keita #1009.json
Keita #1010.json
Keita #1011.json
Keita #1012.json
Keita #1013.json
Keita #1014.json
Keita #1015.json
Keita #1016.json
Keita #1017.json
Keita #1018.json
Keita #1019.json
Keita #1020.json
Keita #12011.json
Keita #12012.json
Keita #12013.json
Keita #12014.json
Keita #12015.json
Keita #12016.json
Keita #12017.json
Keita #12018.json
Keita #12019.json
Keita #12020.json
Keita #2001.json
Keita #2002.json
Keita #2003.json
Keita #2004.json
Keita #2005.json
Keita #2006.json
Keita #2007.json
Keita #2008.json
Keita #2009.json
Keita #2010.json
Keita #2011.json
Keita #2012.json
Keita #2013.json
Keita #2014.json
Keita #2015.json
Keita #2016.json
Keita #2017.json
Keita #2018.json
Keita #2019.json
Keita #2020.json
Keita #3001.json
Keita #3002.json
Keita #3003.json
Keita #3004.json
Keita #3005.json
Keita #3006.json
Keita #3007.json
Keita #3008.json
Keit

In [5]:
#split to presynapse and vesicle json file
unit1 ="vecall0"
unit_syn = "syn0"
unit_mit = "mit0"

sum_train_dir = wsl_pyf+"/"+ "train_label_all0/"
sum_test_dir = wsl_pyf+"/"+ "test_label_all0/"

pre_train = wsl_pyf+"/"+ "train_label_{}".format(unit1)
pre_test  = wsl_pyf+"/"+ "test_label_{}".format(unit1)
pre_train_syn = wsl_pyf+"/"+ "train_label_{}".format(unit_syn)
pre_test_syn  = wsl_pyf+"/"+ "test_label_{}".format(unit_syn)
pre_train_mit = wsl_pyf+"/"+ "train_label_{}".format(unit_mit)
pre_test_mit  = wsl_pyf+"/"+ "test_label_{}".format(unit_mit)
os.makedirs(pre_train, exist_ok=True)
os.makedirs(pre_test, exist_ok=True)
os.makedirs(pre_train_syn, exist_ok=True)
os.makedirs(pre_test_syn, exist_ok=True)
os.makedirs(pre_train_mit, exist_ok=True)
os.makedirs(pre_test_mit, exist_ok=True)

# os.makedirs(sum_train_dir, exist_ok=True)
# os.makedirs(sum_test_dir, exist_ok=True)

for i, fname in enumerate(os.listdir(sum_train_dir)):
    # if i>0:
    #     continue
    print(fname)
    with open(sum_train_dir + "/" + fname, 'rb') as f:
        data = json.load(f)

    base = deepcopy(data)
    base.pop("shapes", None)

    shapes = data.get("shapes", [])
    # filtering
    syn_az_labels = {"synapse", "az"}
    vec_labels = {"vesicle"}
    mit_labels = {"mit"}

    syn_az_shapes = [s for s in shapes if s.get("label") in syn_az_labels]
    vec_shapes = [s for s in shapes if s.get("label") in vec_labels]
    mit_shapes = [s for s in shapes if s.get("label") in mit_labels]

    # 出力データを構築 ("shapes" 以外は全て同じ)
    syn_az_data = deepcopy(base)
    syn_az_data["shapes"] = syn_az_shapes

    vec_data = deepcopy(base)
    vec_data["shapes"] = vec_shapes
    
    mit_data = deepcopy(base)
    mit_data["shapes"] = mit_shapes

    with open(pre_train + "/" + fname, 'w') as f:
        json.dump(vec_data, f)
    with open(pre_train_syn + "/" + fname, 'w') as f:
        json.dump(syn_az_data, f)
    with open(pre_train_mit + "/" + fname, 'w') as f:
        json.dump(mit_data, f)


for i, fname in enumerate(os.listdir(sum_test_dir)):
    # if i>0:
    #     continue
    print(fname)
    with open(sum_test_dir + "/" + fname, 'rb') as f:
        data = json.load(f)

    base = deepcopy(data)
    base.pop("shapes", None)

    shapes = data.get("shapes", [])
    # filtering
    syn_az_labels = {"synapse", "az"}
    vec_labels = {"vesicle"}
    mit_labels = {"mit"}

    syn_az_shapes = [s for s in shapes if s.get("label") in syn_az_labels]
    vec_shapes = [s for s in shapes if s.get("label") in vec_labels]
    mit_shapes = [s for s in shapes if s.get("label") in mit_labels]

    # 出力データを構築 ("shapes" 以外は全て同じ)
    syn_az_data = deepcopy(base)
    syn_az_data["shapes"] = syn_az_shapes

    vec_data = deepcopy(base)
    vec_data["shapes"] = vec_shapes

    mit_data = deepcopy(base)
    mit_data["shapes"] = mit_shapes

    with open(pre_test + "/" + fname, 'w') as f:
        json.dump(vec_data, f)
    with open(pre_test_syn + "/" + fname, 'w') as f:
        json.dump(syn_az_data, f)

    with open(pre_test_mit + "/" + fname, 'w') as f:
        json.dump(mit_data, f)

Keita #1003.json
Keita #1004.json
Keita #1005.json
Keita #1006.json
Keita #1007.json
Keita #1008.json
Keita #1009.json
Keita #1010.json
Keita #2003.json
Keita #2004.json
Keita #2005.json
Keita #2006.json
Keita #2007.json
Keita #2008.json
Keita #2009.json
Keita #2010.json
Keita #3003.json
Keita #3004.json
Keita #3005.json
Keita #3006.json
Keita #3007.json
Keita #3008.json
Keita #3009.json
Keita #3010.json
Keita_#12003.json
Keita_#12004.json
Keita_#12005.json
Keita_#12006.json
Keita_#12007.json
Keita_#12008.json
Keita_#12009.json
Keita_#12010.json
Keita_#4003.json
Keita_#4004.json
Keita_#4005.json
Keita_#4006.json
Keita_#4007.json
Keita_#4008.json
Keita_#4009.json
Keita_#4010.json
Keita_#5003.json
Keita_#5004.json
Keita_#5005.json
Keita_#5006.json
Keita_#5007.json
Keita_#5008.json
Keita_#5009.json
Keita_#5010.json
Keita_#6003.json
Keita_#6004.json
Keita_#6005.json
Keita_#6006.json
Keita_#6007.json
Keita_#6008.json
Keita_#6009.json
Keita_#6010.json
Keita_#9003.json
Keita_#9004.json
Keita_

In [9]:
#merge json file with pre-training

import os
import shutil
# unit ="vecall"

# コピー元ディレクトリ
pre_train = wsl_pyf+"/"+ "train_label_all"
pre_test  = wsl_pyf+"/"+ "test_label_all"
applied   = wsl_pyf+"/"+ "apply_json_curated"
# コピー先ディレクトリ
new_fol = wsl_pyf+"/"+ "labeled_all2"

# コピー先を作成
os.makedirs(new_fol, exist_ok=True)

# コピー元リスト
src_dirs = [pre_train, pre_test] #, applied]

for src in src_dirs:
    if not os.path.isdir(src):
        print(f"Warning: '{src}' は存在しないかディレクトリではありません。スキップします。")
        continue

    for fname in os.listdir(src):
        src_path = os.path.join(src, fname)
        # ファイルのみコピー
        if not os.path.isfile(src_path):
            continue

        dst_path = os.path.join(new_fol, fname)
      
        # if os.path.exists(dst_path):
        #     base, ext = os.path.splitext(fname)
        #     # i = 1
        #     while True:
        #         new_name = f"{base}{ext}"
        #         new_dst = os.path.join(new_fol, new_name)
        #         if not os.path.exists(new_dst):
        #             dst_path = new_dst
        #             break
        #         # i += 1

        shutil.copy2(src_path, dst_path)
        print(f"Copied: {src_path} -> {dst_path}")

print("Done.")

Copied: C:\Users\u9635\pyfolder\EM_analysis//train_label_all\Keita #1003.json -> C:\Users\u9635\pyfolder\EM_analysis//labeled_all2\Keita #1003.json
Copied: C:\Users\u9635\pyfolder\EM_analysis//train_label_all\Keita #1004.json -> C:\Users\u9635\pyfolder\EM_analysis//labeled_all2\Keita #1004.json
Copied: C:\Users\u9635\pyfolder\EM_analysis//train_label_all\Keita #1005.json -> C:\Users\u9635\pyfolder\EM_analysis//labeled_all2\Keita #1005.json
Copied: C:\Users\u9635\pyfolder\EM_analysis//train_label_all\Keita #1006.json -> C:\Users\u9635\pyfolder\EM_analysis//labeled_all2\Keita #1006.json
Copied: C:\Users\u9635\pyfolder\EM_analysis//train_label_all\Keita #1007.json -> C:\Users\u9635\pyfolder\EM_analysis//labeled_all2\Keita #1007.json
Copied: C:\Users\u9635\pyfolder\EM_analysis//train_label_all\Keita #1008.json -> C:\Users\u9635\pyfolder\EM_analysis//labeled_all2\Keita #1008.json
Copied: C:\Users\u9635\pyfolder\EM_analysis//train_label_all\Keita #1009.json -> C:\Users\u9635\pyfolder\EM_anal

In [6]:
#remove_small_polygons
# k_fol =  wsl_pyf+"/"+ "kinoshita"
# y_fol =  wsl_pyf+"/"+ "yamamoto"

k_fol_bl =  wsl_pyf+"/"+ "blind_kinoshita"
y_fol_bl =  wsl_pyf+"/"+ "blind_yamamoto"

train_all_fol =  wsl_pyf+"/"+ "train_label_all0"
test_all_fol =  wsl_pyf+"/"+ "test_label_all0"

all_fol =  wsl_pyf+"/"+ "labeled_all0"

for i, fol in enumerate(os.listdir(k_fol_bl)):
    # if i > 0:
    #     continue
    for j, fname in enumerate(os.listdir(k_fol_bl + "/" + fol)):
        # if j> 0:
        #      continue
        # print(fname)
        # remove_small_polygons(k_fol_bl + "/" + fol + "/" + fname, k_fol_bl + "/" + fol + "/" + fname.replace(".json","")+"_2.json")
        remove_small_polygons(k_fol_bl + "/" + fol + "/" + fname, k_fol_bl + "/" + fol + "/" + fname)
        print(k_fol_bl + "/" + fol + "/" + fname)

for i, fol in enumerate(os.listdir(y_fol_bl)):
    # if i > 0:
    #     continue
    for j, fname in enumerate(os.listdir(y_fol_bl + "/" + fol)):
        # if j> 0:
        #      continue
        # print(fname)
        # remove_small_polygons(y_fol_bl + "/" + fol + "/" + fname, y_fol_bl + "/" + fol + "/" + fname.replace(".json","")+"_2.json")
        remove_small_polygons(y_fol_bl + "/" + fol + "/" + fname, y_fol_bl + "/" + fol + "/" + fname)
        print(y_fol_bl + "/" + fol + "/" + fname)


for i, fname in enumerate(os.listdir(train_all_fol)):
        # print(fname)
        remove_small_polygons(train_all_fol + "/" + fname, train_all_fol + "/" + fname)
        print(train_all_fol + "/" + fname)

for i, fname in enumerate(os.listdir(test_all_fol)):
        # print(fname)
        remove_small_polygons(test_all_fol + "/" + fname, test_all_fol + "/" + fname)
        print(test_all_fol + "/" + fname)

for i, fname in enumerate(os.listdir(all_fol)):
        # print(fname)
        remove_small_polygons(all_fol + "/" + fname, all_fol + "/" + fname)
        print(all_fol + "/" + fname)

C:\Users\u9635\pyfolder\EM_analysis//labeled_all0/Keita #1001.json
C:\Users\u9635\pyfolder\EM_analysis//labeled_all0/Keita #1002.json
C:\Users\u9635\pyfolder\EM_analysis//labeled_all0/Keita #1003.json
C:\Users\u9635\pyfolder\EM_analysis//labeled_all0/Keita #1004.json
C:\Users\u9635\pyfolder\EM_analysis//labeled_all0/Keita #1005.json
C:\Users\u9635\pyfolder\EM_analysis//labeled_all0/Keita #1006.json
C:\Users\u9635\pyfolder\EM_analysis//labeled_all0/Keita #1007.json
C:\Users\u9635\pyfolder\EM_analysis//labeled_all0/Keita #1008.json
C:\Users\u9635\pyfolder\EM_analysis//labeled_all0/Keita #1009.json
C:\Users\u9635\pyfolder\EM_analysis//labeled_all0/Keita #1010.json
C:\Users\u9635\pyfolder\EM_analysis//labeled_all0/Keita #1011.json
C:\Users\u9635\pyfolder\EM_analysis//labeled_all0/Keita #1012.json
C:\Users\u9635\pyfolder\EM_analysis//labeled_all0/Keita #1013.json
C:\Users\u9635\pyfolder\EM_analysis//labeled_all0/Keita #1014.json
C:\Users\u9635\pyfolder\EM_analysis//labeled_all0/Keita #1015.

In [4]:
#convert empty json file from tif image 
import os
import glob
import json
import base64
from PIL import Image

img_dir  = wsl_pyf + "/" + "train"
json_dir =  wsl_pyf + "/" + "pre_json"
os.makedirs(json_dir, exist_ok=True)

for img_path in glob.glob(os.path.join(img_dir, "*.tif")):
    # 1) 画像の base64 化
    with open(img_path, "rb") as f:
        img_b64 = base64.b64encode(f.read()).decode("utf-8")

    # 2) 画像サイズ読み込み
    with Image.open(img_path) as im:
        width, height = im.size

    # 3) JSON スケルトン作成
    data = {
        "version": "5.6.1",
        "flags": {},
        "shapes": [],        # あとで GUI で polygon を追加する
        "imagePath": os.path.basename(img_path),
        "imageData": img_b64,
        "imageHeight": height,
        "imageWidth": width
    }

    # 4) 書き出し
    base = os.path.splitext(os.path.basename(img_path))[0]
    out_file = os.path.join(json_dir, base + ".json")
    with open(out_file, "w", encoding="utf-8") as fw:
        json.dump(data, fw, ensure_ascii=False, indent=2)

    print("Generated:", out_file)

Generated: C:\Users\u9635\pyfolder\EM_analysis//pre_json\Keita #1001.json
Generated: C:\Users\u9635\pyfolder\EM_analysis//pre_json\Keita #1002.json
Generated: C:\Users\u9635\pyfolder\EM_analysis//pre_json\Keita #1003.json
Generated: C:\Users\u9635\pyfolder\EM_analysis//pre_json\Keita #1004.json
Generated: C:\Users\u9635\pyfolder\EM_analysis//pre_json\Keita #1005.json
Generated: C:\Users\u9635\pyfolder\EM_analysis//pre_json\Keita #1006.json
Generated: C:\Users\u9635\pyfolder\EM_analysis//pre_json\Keita #1007.json
Generated: C:\Users\u9635\pyfolder\EM_analysis//pre_json\Keita #1008.json
Generated: C:\Users\u9635\pyfolder\EM_analysis//pre_json\Keita #1009.json
Generated: C:\Users\u9635\pyfolder\EM_analysis//pre_json\Keita #1010.json
Generated: C:\Users\u9635\pyfolder\EM_analysis//pre_json\Keita #1011.json
Generated: C:\Users\u9635\pyfolder\EM_analysis//pre_json\Keita #1012.json
Generated: C:\Users\u9635\pyfolder\EM_analysis//pre_json\Keita #1013.json
Generated: C:\Users\u9635\pyfolder\EM_

In [ ]:
img_fol = r"C:\Users\u9635\pyfolder\EM_analysis\train_pre/"
new_fol = r"C:\Users\u9635\pyfolder\EM_analysis\train/"
os.makedirs(new_fol, exist_ok=True)

target_size = (4000, 2626)  # 最大幅・最大高さ

for n, f in enumerate(os.listdir(img_fol)):
    
    # if n >0:
    #     continue
    # if os.path.exists(new_fol + "/"+f):
    #     continue
        
    fname = img_fol + "/" + f 
    img = tifffile.imread(fname)
# cv2.imshow('Image', img)
    # plt.gray()
    # plt.imshow(img)
    # plt.show()
    crop_height = 361 #train_pre 362, 36 #Ilaria1 361  #int(img.shape[0]/4)
    img =  img[:img.shape[0] - crop_height, :]
    print(img.shape)
   

    img = Image.fromarray(img)
    # # # リサイズ
    img = img.resize((new_w, new_h), resample=Image.LANCZOS)
    img  = np.array(img)
    
    print(img.shape)

    
    # plt.imshow(img)
    # plt.show()
    # min_val = np.min(img)
    # max_val = np.max(img)

# Min-Maxスケーリングを使って0-1の範囲に正規化
    # img = (img - min_val) / (max_val - min_val)

    
  
# 正規化 (0-1の範囲に変換)
    # img = (img - mean) / std

    # plt.imshow(img)
    # plt.show()

    tifffile.imwrite(new_fol + "/"+f, img)
    # img = Image.fromarray(img)
    # img.save(new_fol + "/"+f.split(".")[0] + "_l.png")
    # img.save(new_fol + "/"+f.split(".")[0] + "_l.jpg")
    